# PixelCNN 架构**简析**

本篇主要服务于 VQVAE，所以**不会很详细**，接下来会挑选 Gated PixelCNN 进行讲解，它的核心架构如下图所示:

<div align="center">
    <img src="./imgs/GatedPixelCNN_architecture.png" alt="GatedPixelCNN Architecture" style="width: 600px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/632209862)

[Gated PixelCNN 论文](https://arxiv.org/pdf/1606.05328)

[Gated PixelCNN 模型 Pytorch 实现](https://github.com/SingleZombie/DL-Demos/blob/master/dldemos/pixelcnn/model.py)

## PixelCNN

### 概述
PixelCNN 就是一个**自回归**图像生成模型，在生成第(i, j)个像素时，只能看到他之前的(也就是它左边和上面的)像素，可以类比一下 Transformer 的解码器，先从第一个字开始生成，根据第一个字生成第二个字，再根据前两个字生成第三个字......，直到生成结束符为止；PixelCNN 也是一样的，先生成第一个像素，再根据第一个像素的内容生成第二个像素，根据前两个像素的内容生成第三个像素......，直到生成了完整的图像

生成过程可以建模成下面的式子:

$$
p(x) = \prod_{i, j} p(x_{i, j}|x_{< i, j})
$$

### 掩码卷积

那么怎么保证每个像素只能看到它之前的像素呢？那就需要**掩码卷积**(**Masked Convolution**)了，我们可以使用类似下面这样的卷积核，这样中心像素就只能接收到它之前的像素的信息了，就实现了类似 Transformer 中把未来的词掩盖住的作用，模型的每个像素包含的信息就只能来自于它之前的像素了:

<div align="center">
    <div style="display: flex; justify-content: center; gap: 40px;">
        <div>
            <img src="./imgs/masked_kernel_A.png" alt="masked_kernel_A" style="width: 100px; height: auto;">
            <p style="margin: 4px 0 0;">A 类掩码</p>
        </div>
        <div>
            <img src="./imgs/masked_kernel_B.png" alt="masked_kernel_B" style="width: 100px; height: auto;">
            <p style="margin: 4px 0 0;">B 类掩码</p>
        </div>
    </div>
</div>

通过一次 A 类核(看不到自身信息)和多次 B 类核的作用，就能让每个像素的感受野(可以理解为卷积时给该像素的值有贡献的所有像素)只局限在它之前的像素，如下图所示(以中心像素的感受野为例):

<div align="center">
    <img src="./imgs/perspective_field_origin.png" alt="perspective_field_origin" style="width: 100px; height: auto;">
</div>

## GatedPixelCNN

如果你细心就会发现，其实中心像素的感受野丢失了一部分已知的信息，如下图所示，这个也是原始 PixelCNN 的缺陷，为了改进它，就诞生了 **Gated PixelCNN**:

<div align="center">
    <img src="./imgs/weakness.png" alt="weakness" style="width: 100px; height: auto;">
</div>

### 垂直掩码卷积和水平掩码卷积
作者通过组合垂直掩码卷积和水平掩码卷积来消除原始 PixelCNN 丢失的感受野，如下图所示:

<div align="center">
    <div style="display: flex; justify-content: center; align-items: center; gap: 40px;">
        <div>
            <img src="./imgs/ver_kernel.png" alt="ver_kernel" style="width: 100px; height: auto;">
            <p style="margin: 4px 0 0;">垂直掩码</p>
        </div>
        <div>
            <img src="./imgs/hori_kernel_A.png" alt="hori_kernel_A" style="width: 100px; height: auto;">
            <p style="margin: 4px 0 0;">水平 A 类掩码</p>
        </div>
        <div>
            <img src="./imgs/hori_kernel_B.png" alt="hori_kernel_B" style="width: 100px; height: auto;">
            <p style="margin: 4px 0 0;">水平 B 类掩码</p>
        </div>
    </div>
</div>

怎么做呢？还是以中心像素(i, j)为例，我们可以知道，(i-1, j)垂直掩码卷积之后的感受野和中心像素水平掩码卷积(A)之后的感受野，那么为了得到中心像素之上包含所有像素的感受野，我们就把卷积后(i-1, j)的值加到中心像素上，那么就能把包含上面像素的感受野传递下去了，如下图所示:

<div align="center">
    <img src="./imgs/concat_perspective_field.png" alt="concat_perspective_field" style="width: 450px; height: auto;">
</div>

之后重复(水平掩码卷积改为 B 类)，就可以得到中心像素包含之前所有像素的感受野:

<div align="center">
    <img src="./imgs/full_perspective_field.png" alt="full_perspective_field" style="width: 450px; height: auto;">
</div>


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class VerticalMaskConv2d(nn.Module):
    def __init__(self, *args, **kwags):
        super().__init__()

        self.conv = nn.Conv2d(*args, **kwags)
        # 只保留卷积核的上半部分，下半部分置为0
        h, w = self.conv.weight.shape[-2:]
        mask = torch.zeros((h, w), dtype=torch.float32)
        mask[0:h // 2 + 1] = 1
        mask = mask.reshape((1, 1, h, w))
        self.register_buffer('mask', mask, False)  # 不训练

    def forward(self, x:torch.Tensor):
        self.conv.weight.data *= self.mask
        conv_res = self.conv(x)

        return conv_res

class HorizontalMaskConv2d(nn.Module):
    def __init__(self, conv_type, *args, **kwargs):
        super().__init__()
        assert conv_type in ('A', 'B')

        self.conv = nn.Conv2d(*args, **kwargs)
        # 只保留卷积核中间的左半部分
        h, w = self.conv.weight.shape[-2:]
        mask = torch.zeros((h, w), dtype=torch.float32)
        mask[h // 2, 0:w // 2] = 1  # A类不能看到中间像素(自身)
        if conv_type == 'B':  # B类能看到中间像素(自身)
            mask[h // 2, w // 2] = 1
        mask = mask.reshape((1, 1, h, w))
        self.register_buffer('mask', mask, False)
         
    def forward(self, x:torch.Tensor):
        self.conv.weight.data *= self.mask
        conv_res = self.conv(x)

        return conv_res
    

### 门控机制

Gated PixelCNN 还引入了**门控机制**(也就是最开始的那张图)，可以总结成如下公式:

$$
y = tanh(W_f * x) \odot \sigma (W_g * x)
$$

其中$x$为水平掩码卷积或者垂直掩码卷积的结果，$tanh(W_f * x)$输出范围在[-1, 1]，用于激活特征信息；$\sigma (W_g * x)$输出范围在[0, 1]，用于控制激活的特征信息输出的多少


In [2]:
class GatedBlock(nn.Module):
    def __init__(self, conv_type, in_channels, p, bn=True):
        """
        - `conv_type`: 水平卷积核类型: A or B
        - `in_channels`: 输入通道数
        - `p`: 隐藏层通道数
        - `bn`: 是否BatchNorm
        """
        super().__init__()

        self.conv_type = conv_type
        self.p = p

        self.v_conv = VerticalMaskConv2d(in_channels, 2 * p, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(2 * p) if bn else nn.Identity()
        self.v_to_h_conv = nn.Conv2d(2 * p, 2 * p, 1)
        self.bn2 = nn.BatchNorm2d(2 * p) if bn else nn.Identity()
        self.h_conv = HorizontalMaskConv2d(conv_type, in_channels, 2 * p, 3, 1, 1)
        self.bn3 = nn.BatchNorm2d(2 * p) if bn else nn.Identity()
        self.h_output_conv = nn.Conv2d(p, p, 1)
        self.bn4 = nn.BatchNorm2d(p) if bn else nn.Identity()

    def forward(self, v_input:torch.Tensor, h_input:torch.Tensor):
        # ---------------------------------------------------------------------
        # 垂直掩码卷积核卷积，只看到上半部分的信息
        # ---------------------------------------------------------------------
        v = self.v_conv(v_input)
        v = self.bn1(v)
        # TODO: 整体下移一行，可以理解成把上半部分的信息传递给当前水平流
        v_to_h = v[:, :, 0:-1]  # 去除最后一行
        v_to_h = F.pad(v_to_h, (0, 0, 1, 0))  # 第一行填0
        v_to_h = self.v_to_h_conv(v_to_h)

        v_to_h = self.bn2(v_to_h)

        # ---------------------------------------------------------------------
        # 垂直流门控
        # ---------------------------------------------------------------------
        v1, v2 = v[:, :self.p], v[:, self.p:]
        v1 = torch.tanh(v1)
        v2 = torch.sigmoid(v2)
        v = v1 * v2

        # ---------------------------------------------------------------------
        # 水平掩码卷积核卷积，只看到左半部分的信息
        # ---------------------------------------------------------------------
        h = self.h_conv(h_input)
        h = self.bn3(h)
        h = h + v_to_h

        # ---------------------------------------------------------------------
        # 水平流门控+输出
        # ---------------------------------------------------------------------
        h1, h2 = h[:, :self.p], h[:, self.p:]
        h1 = torch.tanh(h1)
        h2 = torch.sigmoid(h2)
        h = h1 * h2
        h = self.h_output_conv(h)
        h = self.bn4(h)
        if self.conv_type == 'B':  # 第一层不做残差连接
            h = h + h_input

        return v, h
    

### 最终模型

最终的模型如下面代码块所示:

In [5]:
class GatedPixelCNN(nn.Module):
    def __init__(self, n_blocks, p, linear_dim, bn=True, color_level=256):
        super().__init__()
        
        self.block1 = GatedBlock('A', 1, p, bn)
        self.blocks = nn.ModuleList()
        for _ in range(n_blocks):
            self.blocks.append(GatedBlock('B', p, p, bn))
        self.relu = nn.ReLU()
        self.linear1 = nn.Conv2d(p, linear_dim, 1)
        self.linear2 = nn.Conv2d(linear_dim, linear_dim, 1)
        self.out = nn.Conv2d(linear_dim, color_level, 1)

    def forward(self, x:torch.Tensor):
        # input: [bs, c, h, w] -> [bs, p, h, w]*2
        v, h = self.block1(x, x)

        # 形状不变
        for block in self.blocks:
            v, h = block(v, h)

        # 最终输出要h(水平流) -> [bs, linear_dim, h, w]
        x = self.relu(h)
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.out(x)

        # -> [bs, color_level, h, w]
        return x
    

In [7]:
# -------------------------------------------------------------------------------------
# Constant
# -------------------------------------------------------------------------------------
BATCH_SIZE = 4
IMG_SIZE = 64
INPUT_DIM = 3
HIDDEN_DIM = 128


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pixelcnn = GatedPixelCNN(
    n_blocks=15,
    p=HIDDEN_DIM,
    linear_dim=256,
    bn=True,
    color_level=256
).to(device)

img = torch.randint(0, 256, (BATCH_SIZE, 1, IMG_SIZE, IMG_SIZE)).to(device).float()
print(f'Input img size: {img.size()}')
logits = pixelcnn(img)
print(f'Output logits size: {logits.size()}')


Input img size: torch.Size([4, 1, 64, 64])
Output logits size: torch.Size([4, 256, 64, 64])
